# Verificación: construcción de cohorte y etiquetas L1 / L2 / L3

Se construyen las tres etiquetas de resultado clínico directamente desde las tablas MIMIC-IV-ED, sin dependencias de módulos externos. El código es autocontenido para facilitar su revisión y reproducción independiente.

## Configuración y carga de datos

Se inicializa el logging, se carga la ruta de datos desde `.env` y se leen las cuatro tablas necesarias para construir la cohorte y las etiquetas.

In [1]:
import os
import logging
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s — %(message)s", force=True)

load_dotenv(dotenv_path=Path("../../.env"), override=True)
if not os.getenv("MIMIC_IV_ED_PATH"):
    load_dotenv(dotenv_path=Path(".env"), override=True)

DATA = Path(os.getenv("MIMIC_IV_ED_PATH", ""))
print(f"DATA: {DATA}  |  exists: {DATA.exists()}")

DATA: C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data  |  exists: True


In [2]:
df_stays     = pd.read_csv(DATA / "edstays.csv",   low_memory=False, parse_dates=["intime","outtime"])
df_triage    = pd.read_csv(DATA / "triage.csv",    low_memory=False)
df_diagnosis = pd.read_csv(DATA / "diagnosis.csv", low_memory=False)
df_pyxis     = pd.read_csv(DATA / "pyxis.csv",     low_memory=False)

print(f"edstays   : {df_stays.shape}")
print(f"triage    : {df_triage.shape}")
print(f"diagnosis : {df_diagnosis.shape}")
print(f"pyxis     : {df_pyxis.shape}")

edstays   : (425087, 9)
triage    : (425087, 11)
diagnosis : (899050, 6)
pyxis     : (1586053, 7)


## Criterios de inclusión de la cohorte

Se excluyen episodios sin triaje completo (LWBS y ELOPED) y aquellos sin ninguna constante vital registrada. El filtro de edad ≥18 no se aplica porque MIMIC-IV-ED no expone la edad directamente.

In [3]:
VITAL_COLS = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]
EXCLUDE    = {"LEFT WITHOUT BEING SEEN", "ELOPED"}

n_total = len(df_stays)

# Exclusión de episodios sin evaluación clínica (LWBS y ELOPED)
edstays = df_stays[~df_stays["disposition"].isin(EXCLUDE)].copy()
print(f"Tras excluir LWBS/ELOPED: {len(edstays):,}  (eliminados: {n_total - len(edstays):,})")

# Merge con triage para añadir constantes vitales
cohort = edstays.merge(df_triage, on="stay_id", how="left", suffixes=("", "_triage"))
if "subject_id_triage" in cohort.columns:
    cohort = cohort.drop(columns=["subject_id_triage"])

# Exclusión de episodios sin ninguna constante vital en triage
vital_present = [c for c in VITAL_COLS if c in cohort.columns]
has_vital  = cohort[vital_present].notna().any(axis=1)
n_before   = len(cohort)
cohort     = cohort[has_vital].copy()
print(f"Tras exigir >=1 vital: {len(cohort):,}  (eliminados: {n_before - len(cohort):,})")

# Filtro de edad: columna no disponible en MIMIC-IV-ED — limitación conocida del dataset
age_cols = [c for c in cohort.columns if "age" in c.lower()]
if age_cols:
    n_before = len(cohort)
    cohort = cohort[pd.to_numeric(cohort[age_cols[0]], errors="coerce") >= 18].copy()
    print(f"Tras filtro edad >=18: {len(cohort):,}")
else:
    print("Columna de edad no encontrada — filtro >=18 OMITIDO (limitación conocida)")

print(f"\nTamaño final de cohorte: {len(cohort):,} stays")

Tras excluir LWBS/ELOPED: 413,222  (eliminados: 11,865)
Tras exigir >=1 vital: 397,601  (eliminados: 15,621)
Columna de edad no encontrada — filtro >=18 OMITIDO (limitación conocida)

Tamaño final de cohorte: 397,601 stays


## Limpieza de outliers fisiológicamente imposibles en vitales de triage

Los CSVs crudos de MIMIC-IV-ED contienen errores de entrada (ej. DBP=661.672 mmHg) que destruyen la correlación entre variables y degradan el rendimiento de los modelos. Se aplican umbrales de imposibilidad fisiológica — no rangos clínicos normales — convirtiendo los valores fuera de rango a NaN para su posterior imputación por mediana en cada modelo.

In [4]:
# Umbrales de imposibilidad fisiológica (fuera de estos rangos = error de entrada → NaN)
# NOTA: temperature en MIMIC-IV-ED está en Fahrenheit (rango típico 95-105 °F)
VITAL_BOUNDS = {
    "temperature": (85.0,  115.0),  # °F  — fuera: incompatible con vida
    "heartrate":   (0.0,   300.0),  # bpm
    "resprate":    (0.0,    60.0),  # rpm
    "o2sat":       (50.0,  100.0),  # %
    "sbp":         (0.0,   300.0),  # mmHg
    "dbp":         (0.0,   200.0),  # mmHg
}

print("Outliers fisiológicamente imposibles convertidos a NaN:")
for col, (lo, hi) in VITAL_BOUNDS.items():
    if col not in cohort.columns:
        continue
    mask = cohort[col].notna() & ((cohort[col] < lo) | (cohort[col] > hi))
    n = mask.sum()
    cohort.loc[mask, col] = float("nan")
    print(f"  {col:12s} [{lo}, {hi}]: {n:,} valores → NaN")

print(f"\nCorrelación SBP-DBP tras limpieza: {cohort[['sbp','dbp']].corr().iloc[0,1]:.4f}  (esperado ~0.50-0.65)")

Outliers fisiológicamente imposibles convertidos a NaN:
  temperature  [85.0, 115.0]: 521 valores → NaN
  heartrate    [0.0, 300.0]: 7 valores → NaN
  resprate     [0.0, 60.0]: 38 valores → NaN
  o2sat        [50.0, 100.0]: 137 valores → NaN
  sbp          [0.0, 300.0]: 17 valores → NaN
  dbp          [0.0, 200.0]: 430 valores → NaN

Correlación SBP-DBP tras limpieza: 0.4965  (esperado ~0.50-0.65)


## L1 — Ingreso hospitalario (`hospital_admission`)

Se define como cualquier episodio con disposición ADMITTED o TRANSFER. Representa la tarea de predicción principal (prevalencia ~39%).

In [5]:
ADMIT = {"ADMITTED", "TRANSFER"}
cohort["L1"] = cohort["disposition"].isin(ADMIT).astype(int)
print(f"L1 positivos: {cohort['L1'].sum():,}  ({cohort['L1'].mean()*100:.2f}%)")

L1 positivos: 154,006  (38.73%)


## L2 — Resultado crítico en urgencias (`critical_ed_outcome`)

Se define por presencia de códigos ICD de alta mortalidad (sepsis, fallo respiratorio, shock, IAM, ictus, PCR) o disposición EXPIRED. Prevalencia ~1.5%, clase muy desbalanceada.

In [6]:
L2_ICD10 = ["A41", "J96", "R57", "I21", "I63", "I46"]
L2_ICD9  = ["038", "51881", "7855", "410", "434", "4275"]

cohort_stays = set(cohort["stay_id"])
diag = df_diagnosis[df_diagnosis["stay_id"].isin(cohort_stays)].copy()
diag["_code"] = diag["icd_code"].astype(str).str.replace(".","",regex=False).str.upper().str.strip()
ver = diag["icd_version"].astype(str).str.strip()

pat10 = "|".join(f"^{p}" for p in L2_ICD10)
pat9  = "|".join(f"^{p}" for p in L2_ICD9)
icd_l2 = set(diag.loc[
    ((ver=="10") & diag["_code"].str.match(pat10, na=False)) |
    ((ver=="9")  & diag["_code"].str.match(pat9,  na=False)),
    "stay_id"
])
expired = set(cohort.loc[cohort["disposition"]=="EXPIRED", "stay_id"])

cohort["L2"] = cohort["stay_id"].isin(expired | icd_l2).astype(int)
print(f"L2 positivos: {cohort['L2'].sum():,}  ({cohort['L2'].mean()*100:.2f}%)")
print(f"  -> desde EXPIRED: {len(expired):,}")
print(f"  -> desde ICD:     {len(icd_l2):,}")

L2 positivos: 6,071  (1.53%)
  -> desde EXPIRED: 88
  -> desde ICD:     6,023


## L3 — Intervención crítica en urgencias (`critical_intervention_ed`)

Se define por administración de vasopresores (identificada en `pyxis`) o procedimientos invasivos codificados en ICD (ventilación mecánica, RCP, transfusión masiva). Prevalencia ~0.6%, clase más desbalanceada del conjunto.

In [7]:
L3_ICD10 = ["Z9911", "5A1935Z", "5A1945Z", "I46", "30233N1", "30243N1"]
L3_ICD9  = ["967", "4275", "990"]
VASOPATTERN = "norepinephrine|epinephrine|dopamine|vasopressin|phenylephrine|noradrenaline"

# Identificación de vasopresores dispensados en pyxis durante la estancia en urgencias
pyxis_c = df_pyxis[df_pyxis["stay_id"].isin(cohort_stays)].copy()
vaso_stays = set(pyxis_c.loc[
    pyxis_c["name"].astype(str).str.lower().str.contains(VASOPATTERN, regex=True, na=False),
    "stay_id"
])

# Identificación de procedimientos críticos por ICD
pat10_l3 = "|".join(f"^{p}" for p in L3_ICD10)
pat9_l3  = "|".join(f"^{p}" for p in L3_ICD9)
icd_l3 = set(diag.loc[
    ((ver=="10") & diag["_code"].str.match(pat10_l3, na=False)) |
    ((ver=="9")  & diag["_code"].str.match(pat9_l3,  na=False)),
    "stay_id"
])

cohort["L3"] = cohort["stay_id"].isin(vaso_stays | icd_l3).astype(int)
print(f"L3 positivos: {cohort['L3'].sum():,}  ({cohort['L3'].mean()*100:.2f}%)")
print(f"  -> desde pyxis: {len(vaso_stays):,}")
print(f"  -> desde ICD:   {len(icd_l3):,}")

L3 positivos: 2,269  (0.57%)
  -> desde pyxis: 2,173
  -> desde ICD:   127


## Validación de la cohorte y las etiquetas

Se verifican tasas de positividad, ausencia de solapamiento inesperado entre etiquetas y consistencia de la distribución de disposiciones.

In [8]:
print("=== Tasas de positividad ===")
print(cohort[["L1","L2","L3"]].mean().round(4))

print("\n=== Conteos absolutos ===")
print(cohort[["L1","L2","L3"]].sum())

print("\n=== L1 ∧ L2 ∧ L3 simultáneos ===")
print((cohort[["L1","L2","L3"]].sum(axis=1)==3).sum())

print("\n=== Distribución de disposition en la cohorte ===")
print(cohort["disposition"].value_counts())

=== Tasas de positividad ===
L1    0.3873
L2    0.0153
L3    0.0057
dtype: float64

=== Conteos absolutos ===
L1    154006
L2      6071
L3      2269
dtype: int64

=== L1 ∧ L2 ∧ L3 simultáneos ===
579

=== Distribución de disposition en la cohorte ===
disposition
HOME                           238533
ADMITTED                       147215
TRANSFER                         6791
OTHER                            3141
LEFT AGAINST MEDICAL ADVICE      1833
EXPIRED                            88
Name: count, dtype: int64


In [9]:
print("=== dtypes ===")
print(cohort.dtypes)

=== dtypes ===
subject_id                    int64
hadm_id                     float64
stay_id                       int64
intime               datetime64[us]
outtime              datetime64[us]
gender                          str
race                            str
arrival_transport               str
disposition                     str
temperature                 float64
heartrate                   float64
resprate                    float64
o2sat                       float64
sbp                         float64
dbp                         float64
pain                            str
acuity                      float64
chiefcomplaint                  str
L1                            int64
L2                            int64
L3                            int64
dtype: object


## Exportación a `data/interim/`

Se persiste la cohorte con etiquetas en formato Parquet para su uso en el notebook de split temporal.

In [10]:
out = Path("../../data/interim/cohort_with_labels.parquet")
out.parent.mkdir(parents=True, exist_ok=True)
cohort.to_parquet(out, index=False)
print(f"Guardado: {out.resolve()}  ({cohort.shape})")

Guardado: C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data\interim\cohort_with_labels.parquet  ((397601, 21))
